## To Shift Scale and Combine the Images

In [1]:
# importing all the relevant libraries 
import numpy as np
import astropy
import astroalign as aa
import photutils
import ccdproc
import os
import astropy.io.fits as fits
import matplotlib.pyplot as plt
from ccdproc import CCDData, combiner
from astropy import units as u
from astropy.time import Time
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_2dg, centroid_sources
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.segmentation import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift


In [2]:
# Navigating to folder that contains all the image files - only run this once
# If this block has already run, uncomment the next line and run again
# os.chdir("..")
os.chdir("NGC_2547_sub/")


In [ ]:
# Collating b-band images
images = ccdproc.ImageFileCollection(".", glob_include='*_b.fit')  # collect all b-band images
#for fn in images.files_filtered():  # loop through and print all b-band files
    #print(fn)

scim = []                                                                    # This is an empty list - scim
for fn in images.files_filtered():
    # print(fn)
    scim.append(CCDData.read(fn, unit = "adu"))                              # reads the file and adds it to the list 
    # This code will adds MJD-OBS to the header so it will (later) stop producing error messages
    times = scim[-1].header['DATE-OBS']                                      # Get the time
    t = Time(times, format='isot', scale='utc')                              # Read the time from UTC format
    scim[-1].header['MJD-OBS'] = (float(t.mjd), 'MJD') 

Light_NGC_2547_10.0s_IRCUT_20260331-200329_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200351_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200403_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200415_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200425_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200437_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200500_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200512_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200524_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200534_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200546_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200616_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200627_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200639_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200651_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200702_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200748_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200759_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200811_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200823_b.fit
Light_NGC_2547_10.0s

## Shifting
#### Using the automated alignment to shift all images to the correct orientation 

In [ ]:
# generating new names for the automatically aligned images
newname=[]                                                              # creating a new empty variable
for fn in images.files_filtered():                                      # iterating through all the images
    newname.extend(["aa"+fn])                                           # creating a list of file names with the original filenames and adding "aa" infront
print(newname)                                                          # printing the new names

['aaLight_NGC_2547_10.0s_IRCUT_20260331-200329_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200351_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200403_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200415_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200425_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200437_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200500_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200512_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200524_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200534_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200546_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200616_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200627_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200639_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200651_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200702_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200748_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200759_b.fit', 'aaLight_NGC_2547_10.0s_IRC

In [6]:
# Automated alignment using image no6 as the reference image
refidx = 5
for idx, thisimage in enumerate(scim):
    # Define images to be shifted
    # Plus zero is a quirky fix to an endian issue (old school)
    img_ref=scim[refidx].data+0       # Reference image
    img_example=scim[idx].data+0      # Image to be shifted image
    # print('Reference and example image defined')

    img_aligned, footprint = aa.register(img_example, img_ref, detection_sigma=5.0, min_area=9, fill_value=-99999.99)
    # print('Image alignment determined')

    temp=scim[idx]
    temp.data=img_aligned
    temp.meta = scim[idx].meta

    # Write the image to a file using the new name (aa- prefix)
    print(newname[idx])
    temp.write(newname[idx])


aaLight_NGC_2547_10.0s_IRCUT_20260331-200329_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200351_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200403_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200415_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200425_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200437_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200500_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200512_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200524_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200534_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200546_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200616_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200627_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200639_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200651_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200702_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200748_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200759_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20260331-200811_b.fit
aaLight_NGC_2547_10.0s_IRCUT_20

command prompt lines

ds9 aaLight_NGC* &


# Scaling

In [10]:
# Three positions of bright unsaturated stars
positions = [(603.44,1027), (807,863.08), (499.24, 855.88)]   #x-y notation
apertures = CircularAperture(positions, r=15.0)
phot_table = aperture_photometry(scim[0], apertures)
#print(phot_table)
phot_tables=[]
for idx, thisimage in enumerate(scim):                                  # Comment - summing the fluxes of the apertures of each image 
    phot_tables.extend([aperture_photometry(thisimage, apertures)])      
    #print(idx, phot_tables[idx])


refidx=0 # The index of our reference image. By default it is zero, but should it be different?

# This code is missing comments. What is it doing?
for idx, thisimage in enumerate(scim): 
    print(idx)
    # normalising based on the reference image - scaling to the reference image
    print(phot_tables[refidx]['aperture_sum']/phot_tables[idx]['aperture_sum'])
    print(np.ma.median(phot_tables[refidx]['aperture_sum']/phot_tables[idx]['aperture_sum']))

0
[1. 1. 1.]
1.0
1
[1.11725559 1.16963834 1.01574924]
1.1172555914771731
2
[1.13639392 1.17314315 1.05593773]
1.1363939216776284
3
[1.12320305 1.2062254  1.11367457]
1.1232030541717084
4
[1.21623089 1.19053035 1.17036234]
1.19053034787383
5
[1.23575222 1.23923792 1.17502783]
1.2357522181664287
6
[1.20983128 1.21106501 1.11233278]
1.2098312784483118
7
[1.23721775 1.23202021 1.10199528]
1.2320202102028217
8
[1.25340916 1.26305951 1.1225998 ]
1.2534091638196079
9
[1.22240977 1.26024502 1.18338462]
1.2224097701413752
10
[1.2733373  1.37001071 1.16280219]
1.2733372974426989
11
[1.1760687  1.25880554 1.11531245]
1.1760686976204833
12
[1.168      1.28443017 1.10915689]
1.1679999973149453
13
[1.20398916 1.26253263 1.14372413]
1.2039891564393248
14
[1.15132667 1.28221196 1.13006696]
1.15132667438279
15
[1.1902819  1.27634029 1.13051844]
1.1902819035132062
16
[1.21009649 1.31446191 1.151103  ]
1.2100964936432532
17
[1.3064724  1.32262601 1.19564978]
1.3064723982185207
18
[1.21907715 1.2996631  1

# Combine